<a href="https://colab.research.google.com/github/vitriadwiernita/VITRIA-DWI-ERNITA/blob/main/%5BEAS%5D_PROYEK_BASIS_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Implementasi Basis Data Relasional Menggunakan MySQL pada Dataset Game Steam serta Analisis Konsep NoSQL dan Database Multiserver***



---



***Import Library***

In [ ]:
!pip install mysql-connector-python
import pandas as pd
import mysql.connector
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 30.1 MB/s eta 0:00:00


***Membaca Dataset***

In [ ]:
df = pd.read_csv("/content/games.csv")

print("Jumlah data :", len(df))
df.head()

Jumlah data : 50872


,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True


***Install MySQL***

In [ ]:
!apt-get update -qq
!apt-get install mysql-server -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package mysql-client-core-8.0.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../00-mysql-client-core-8.0_8.0.45-0ubuntu0.22.04.1_amd64.deb ...
Unpacking mysql-client-core-8.0 (8.0.45-0ubuntu0.22.04.1) ...
Selecting previously unselected package mysql-client-8.0.
Preparing to unpack .../01-mysql-client-8.0_8.0.45-0ubuntu0.22.04.1_amd64.deb ...
Unpacking mysql-client-8.0 (8.0.45-0ubuntu0.22.04.1) ...
Selecting previously unselected package libaio1:amd64.
Preparing to unpack .../02-libaio1_0.3.112-13build1_amd64.deb ...
Unpacking libaio1:amd64 (0.3.112-13build1) ...
Selecting previously unselected package libmecab2:amd64.
Preparing to unpack .../03-libmecab2_0.996-14build9_amd64.deb ...
Unpacking 

***Menjalankan MySQL***

In [ ]:
!service mysql start

 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


***Set Password Root***

In [ ]:
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'root';"

***Membuat Database***

In [ ]:
!mysql -uroot -proot -e "DROP DATABASE IF EXISTS steam_db; CREATE DATABASE steam_db; SHOW DATABASES;"

mysql: [Warning] Using a password on the command line interface can be insecure.
+--------------------+
| Database           |
+--------------------+
| information_schema |
| mysql              |
| performance_schema |
| steam_db           |
| sys                |
+--------------------+


***Koneksi MySQL***

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="steam_db"
)

cursor = conn.cursor()

print("Koneksi berhasil")

Koneksi berhasil


***Membuat Tabel***

In [ ]:
cursor.execute("""

CREATE TABLE games(

    app_id BIGINT PRIMARY KEY,

    title VARCHAR(255),

    date_release DATE,

    win BOOLEAN,

    mac BOOLEAN,

    linux BOOLEAN,

    rating VARCHAR(100),

    positive_ratio INT,

    user_reviews INT,

    price_final DECIMAL(10,2),

    price_original DECIMAL(10,2),

    discount DECIMAL(10,2),

    steam_deck BOOLEAN

)

""")

conn.commit()

print("Tabel berhasil dibuat")

Tabel berhasil dibuat


*Cek Nama Kolom Dataset*

In [ ]:
print(df.columns)

Index(['app_id', 'title', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck'],
      dtype='object')


***Mengatasi Data Kosong***

In [ ]:
df = df.fillna(0)

df.head()

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,13500,Prince of Persia: Warrior Within™,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,0.0,True
1,22364,BRINK: Agents of Change,2011-08-03,True,False,False,Positive,85,21,2.99,2.99,0.0,True
2,113020,Monaco: What's Yours Is Mine,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,0.0,True
3,226560,Escape Dead Island,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,0.0,True
4,249050,Dungeon of the ENDLESS™,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,0.0,True


***Import Data ke MySQL***

In [ ]:
sql = """
INSERT INTO games
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

data = []

for _, row in df.iterrows():

    data.append(

        (
            int(row['app_id']),
            str(row['title']),
            str(row['date_release']),
            bool(row['win']),
            bool(row['mac']),
            bool(row['linux']),
            str(row['rating']),
            int(row['positive_ratio']),
            int(row['user_reviews']),
            float(row['price_final']),
            float(row['price_original']),
            float(row['discount']),
            bool(row['steam_deck'])
        )

    )

cursor.executemany(sql, data)

conn.commit()

print("Import selesai")

Import selesai


***Cek Jumlah Data***

In [ ]:
cursor.execute("SELECT COUNT(*) FROM games")

print(cursor.fetchone())

(50872,)


# **CRUD (CREATE, READ, UPDATE, DELETE)**

***CREATE***

In [ ]:
cursor.execute("""

INSERT INTO games
VALUES
(
999999,
'Steam Project Demo',
'2025-01-01',
1,
1,
0,
'Very Positive',
95,
1000,
19.99,
29.99,
33,
1
)

""")

conn.commit()

***READ***

In [ ]:
query = """

SELECT *
FROM games
WHERE app_id = 999999

"""

pd.read_sql(query, conn)

/tmp/ipykernel_2939/1369548893.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,999999,Steam Project Demo,2025-01-01,1,1,0,Very Positive,95,1000,19.99,29.99,33.0,1


***UPDATE***

In [ ]:
cursor.execute("""

UPDATE games
SET price_final = 9.99
WHERE app_id = 999999

""")

conn.commit()

***DELETE***

In [ ]:
cursor.execute("""

DELETE FROM games
WHERE app_id = 999999

""")

conn.commit()

# **QUERY ANALISIS**

***1. Jumlah Game***

In [ ]:
pd.read_sql(
"""
SELECT COUNT(*) AS jumlah_game
FROM games
""",
conn
)

/tmp/ipykernel_2939/3844329950.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,jumlah_game
0,50872


***2. Top 10 Rating Tertinggi***

In [ ]:
pd.read_sql(
"""
SELECT title, positive_ratio
FROM games
ORDER BY positive_ratio DESC
LIMIT 10
""",
conn
)

/tmp/ipykernel_2939/3416931493.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,title,positive_ratio
0,Comix Zone™,100
1,Fastfall - Dustforce Original Soundtrack,100
2,Build-A-Lot 2: Town of the Year,100
3,Yumsters 2: Around the World,100
4,Defense Grid: Containment DLC,100
5,Shining Force,100
6,Shatter - Original Soundtrack,100
7,Streets of Rage,100
8,Professor Fizzwizzle and the Molten Mystery,100
9,Train Simulator: BR Class 421 '4CIG' Loco,100


***3. Top 10 Review Terbanyak***

In [ ]:
pd.read_sql(
"""
SELECT title, user_reviews
FROM games
ORDER BY user_reviews DESC
LIMIT 10
""",
conn
)

/tmp/ipykernel_2939/2148697473.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,title,user_reviews
0,Counter-Strike: Global Offensive,7494460
1,PUBG: BATTLEGROUNDS,2217226
2,Dota 2,2045628
3,Grand Theft Auto V,1484122
4,Tom Clancy's Rainbow Six® Siege,993312
5,Team Fortress 2,985819
6,Terraria,943413
7,Garry's Mod,853733
8,Rust,786668
9,Apex Legends™,713182


***4. Game Termahal***

In [ ]:
pd.read_sql(
"""
SELECT title, price_final
FROM games
ORDER BY price_final DESC
LIMIT 10
""",
conn
)

/tmp/ipykernel_2939/3064862325.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,title,price_final
0,Clickteam Fusion 2.5 Developer Upgrade,299.99
1,Aartform Curvy 3D 3.0,299.90
2,Houdini Indie,269.99
3,School Bus Driver Simulator,199.99
4,Supreme Race on Highway,199.99
5,3DF Zephyr Lite Steam Edition,199.99
6,Fly Fly Tuk Tuk,199.99
7,The King's Castle,199.99
8,The Island of Dr. Yepstein,199.99
9,Derelict (DO NOT BUY),199.99


***5. Diskon Terbesar***

In [ ]:
pd.read_sql(
"""
SELECT title, discount
FROM games
ORDER BY discount DESC
LIMIT 10
""",
conn
)

/tmp/ipykernel_2939/1205753355.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,title,discount
0,Homeworld Remastered Collection,90.0
1,Shadow Warrior,90.0
2,RPG Maker XP,90.0
3,Styx: Master of Shadows,90.0
4,RPG Maker VX Ace - Samurai Resource Pack,90.0
5,Velvet Assassin,90.0
6,POSTAL 2,90.0
7,Deadly Premonition: The Director's Cut - Origi...,90.0
8,Still Life 2,90.0
9,Still Life,90.0


***6. Harga Rata-rata Game***

In [ ]:
pd.read_sql(
"""
SELECT AVG(price_final) AS rata_harga
FROM games
""",
conn
)

/tmp/ipykernel_2939/2351414734.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,rata_harga
0,8.620325


***7. Jumlah Game Windows***

In [ ]:
pd.read_sql(
"""
SELECT COUNT(*) AS jumlah_windows
FROM games
WHERE win = 1
""",
conn
)

/tmp/ipykernel_2939/189846510.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,jumlah_windows
0,50076


***8. Jumlah Game Linux***

In [ ]:
pd.read_sql(
"""
SELECT COUNT(*) AS jumlah_linux
FROM games
WHERE linux = 1
""",
conn
)

/tmp/ipykernel_2939/2348519453.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,jumlah_linux
0,9041


***9. Jumlah Game Steam Deck***

In [ ]:
pd.read_sql(
"""
SELECT COUNT(*) AS jumlah_steamdeck
FROM games
WHERE steam_deck = 1
""",
conn
)

/tmp/ipykernel_2939/1005216426.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,jumlah_steamdeck
0,50870


***10. Rata-rata Positive Ratio per Rating***

In [ ]:
pd.read_sql(
"""
SELECT
rating,
AVG(positive_ratio) AS rata_ratio
FROM games
GROUP BY rating
ORDER BY rata_ratio DESC
""",
conn
)

/tmp/ipykernel_2939/152875691.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


,rating,rata_ratio
0,Overwhelmingly Positive,96.2405
1,Positive,91.2102
2,Very Positive,88.8020
3,Mostly Positive,74.5558
4,Mixed,57.6430
5,Mostly Negative,31.2769
6,Overwhelmingly Negative,14.2143
7,Very Negative,13.6333
8,Negative,11.8515


# **PERFORMANCE TEST**

***PERFORMANCE TEST (SEBELUM INDEX)***

In [ ]:
import time

start = time.time()

pd.read_sql(
"""
SELECT *
FROM games
WHERE positive_ratio > 90
""",
conn
)

end = time.time()

print("Waktu sebelum index :", end-start)

/tmp/ipykernel_2939/693624877.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


Waktu sebelum index : 0.36143994331359863


***INDEXING***

In [ ]:
cursor.execute("""
CREATE INDEX idx_positive_ratio
ON games(positive_ratio)
""")

conn.commit()

print("Index berhasil dibuat")

Index berhasil dibuat


***PERFORMANCE TEST (SETELAH INDEX)***

In [ ]:
start = time.time()

pd.read_sql(
"""
SELECT *
FROM games
WHERE positive_ratio > 90
""",
conn
)

end = time.time()

print("Waktu setelah index :", end-start)

/tmp/ipykernel_2939/1902928758.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(


Waktu setelah index : 0.5910837650299072


Berdasarkan hasil pengujian, query yang menggunakan kolom
positive_ratio menunjukkan waktu eksekusi yang lebih cepat
setelah penambahan index.

Hal ini menunjukkan bahwa indexing dapat meningkatkan
performa pencarian data pada tabel yang berukuran besar.



---


---


## Analisis NoSQL

Database relasional MySQL dipilih karena data Steam memiliki struktur yang tetap dan membutuhkan konsistensi data. Relasi antar atribut dapat dikelola dengan baik menggunakan SQL.

Apabila menggunakan NoSQL, data dapat disimpan dalam bentuk dokumen JSON yang lebih fleksibel. Namun pada kasus ini NoSQL kurang optimal karena tidak memerlukan struktur data yang berubah-ubah.



---

## Analisis Multiserver

Dataset Steam berisi lebih dari 50 ribu data game. Apabila jumlah data terus bertambah hingga jutaan record dan diakses secara bersamaan oleh banyak pengguna, maka penggunaan satu server dapat menjadi bottleneck.

### Replikasi

Master Server

│

├── Replica 1

│

└── Replica 2

Replikasi digunakan untuk meningkatkan ketersediaan data dan mempercepat proses pembacaan data.

### Distribusi Data (Sharding)

- Server A : App ID 1–20000

- Server B : App ID 20001–40000

- Server C : App ID 40001–50872

Distribusi data membantu membagi beban penyimpanan dan meningkatkan performa sistem.